# Notebook 05: Content Moderation at Scale

## Why This Matters

Content moderation is a defining challenge of modern platforms. YouTube processes 500 hours of video per minute; Twitter handles 500M tweets per day; Facebook moderates content in 100+ languages. The stakes are high in both directions: missed harmful content causes real-world harm, while over-moderation damages creator trust and free expression. ML-based moderation is not optional at this scale. But pure ML is also not sufficient - nuanced context, evolving policies, and appeals all require human judgment. Understanding the moderation stack is essential for platform, trust & safety, and ML roles.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
np.random.seed(42)
print("Ready.")

## 1. The Moderation Stack: Latency Tiers

**Tier 1 - Pre-post (synchronous, <200ms)**:
- Block before publication; only highest-severity (CSAM, credible threats)
- Model must be fast: logistic regression on TF-IDF

**Tier 2 - Near-real-time (<1 second)**:
- Content published but immediately queued
- Balance precision and recall; can use small GPU model
- Severity: hate speech, harassment, violent extremism

**Tier 3 - Batch (minutes to hours)**:
- Deep analysis: video understanding, graph patterns
- Full neural models, multi-modal fusion
- Severity: spam, misinformation, coordinated manipulation

**Key insight**: The latency tier determines your model choices. Don't propose a 2B-parameter transformer for Tier 1.

In [ ]:
def generate_moderation_data(n=5000):
    benign = [
        "I really enjoyed {t} today", "Great post about {t}", "Looking forward to {t}",
        "Can anyone recommend a good {t}?", "I disagree with {t} but respect your view",
    ]
    toxic = [
        "I hate {g} so much they should all be removed",
        "You are worthless and nobody wants you",
        "Go kill yourself you stupid {s}",
        "{g} are ruining everything get out of here",
    ]
    topics = ["cooking", "sports", "technology", "music", "travel"]
    groups = ["liberals", "conservatives", "vegans", "gamers"]
    slurs = ["moron", "idiot", "loser"]
    texts, labels = [], []
    n_toxic = int(n * 0.15)
    for _ in range(n - n_toxic):
        tmpl = np.random.choice(benign)
        texts.append(tmpl.format(t=np.random.choice(topics), g=np.random.choice(groups)))
        labels.append(0)
    for _ in range(n_toxic):
        tmpl = np.random.choice(toxic)
        texts.append(tmpl.format(g=np.random.choice(groups), s=np.random.choice(slurs)))
        labels.append(1)
    return pd.DataFrame({"text": texts, "label": labels}).sample(frac=1, random_state=42).reset_index(drop=True)

df = generate_moderation_data(5000)
print(f"Dataset: {len(df)} samples, {df.label.mean():.1%} toxic")

## 2. Precision-Recall Tradeoff in Moderation

The tradeoff is asymmetric:
- **False positives (ham -> spam folder)**: hurt legitimate creators, create appeals, damage trust
- **False negatives (toxic -> published)**: harmful content reaches users, causes harm, regulatory risk

Different platforms make different tradeoffs:
- YouTube: prioritizes recall (catch everything) for advertiser brand safety
- Twitter: historically prioritized precision (protect free speech)
- Children's platforms: near-100% recall regardless of false positive cost

The right threshold depends on: (1) severity of violation, (2) cost of false positive to creators, (3) whether human review is available.

In [ ]:
X = df.text.values; y = df.label.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

tier1_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=10000, min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(C=1.0, class_weight="balanced", solver="lbfgs", max_iter=1000))
])
tier1_pipe.fit(X_train, y_train)
probs = tier1_pipe.predict_proba(X_test)[:, 1]

print("=== Tier 1 Classifier (LR + TF-IDF) ===")
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, probs):.4f}")
print()
print(classification_report(y_test, (probs > 0.5).astype(int), target_names=["benign", "toxic"]))

## 3. Confidence-Based Routing (Active Learning in Production)

Rather than hard classification, production systems route content based on model confidence:
- **High confidence toxic** (score > 0.85): auto-remove
- **Medium confidence** (0.30 < score < 0.85): route to human review queue
- **High confidence benign** (score < 0.30): publish

This IS active learning in production: the uncertain cases that go to human review are the most informative training examples. Over time, the human review queue shrinks as the model becomes more confident.

In [ ]:
def confidence_routing(probs, high_thresh=0.85, low_thresh=0.30):
    decisions = []
    for p in probs:
        if p >= high_thresh: decisions.append("auto_remove")
        elif p <= low_thresh: decisions.append("auto_approve")
        else: decisions.append("human_review")
    return np.array(decisions)

decisions = confidence_routing(probs)
df_dec = pd.DataFrame({"true_label": y_test, "prob": probs, "decision": decisions})
print("=== Confidence-Based Routing ===")
for d in ["auto_remove", "human_review", "auto_approve"]:
    mask = decisions == d
    if mask.sum() > 0:
        toxic_rate = df_dec.loc[mask, "true_label"].mean()
        print(f"  {d:<15}: n={mask.sum():4d} ({mask.mean():.1%}), toxic_rate={toxic_rate:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(probs[y_test==0], bins=50, alpha=0.7, label="Benign", color="blue")
axes[0].hist(probs[y_test==1], bins=50, alpha=0.7, label="Toxic", color="red")
axes[0].axvline(0.30, color="green", linestyle="--", label="Low threshold")
axes[0].axvline(0.85, color="orange", linestyle="--", label="High threshold")
axes[0].set_xlabel("Model Score"); axes[0].set_title("Score Distribution by Class"); axes[0].legend()
routing_counts = pd.Series(decisions).value_counts()
axes[1].bar(routing_counts.index, routing_counts.values)
axes[1].set_title("Routing Distribution")
plt.tight_layout(); plt.savefig("/tmp/moderation_routing.png", dpi=80); plt.show()

## 4. Adversarial Robustness

Adversarial content evolution is constant:
- **Homoglyphs**: replacing letters with look-alikes (h4te -> hate)
- **Leet speak**: substituting numbers/symbols for letters
- **Dog-whistling**: coded language known only to in-group
- **Image text**: embedding slurs in images to evade text classifiers

**Defense strategies**:
- Text normalization: map Unicode variants to canonical forms
- Character-level models: n-gram models catch homoglyphs
- Multilingual models: fine-tune on multilingual data
- Human review feedback loops: escalate novel bypass patterns to model update

**Key principle**: Your model WILL be attacked. Build for adversarial evolution.

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Latency tiers | Pre-post (<200ms), near-real-time (<1s), batch (hours) |
| Confidence routing | Uncertain predictions -> humans; this is active learning |
| False positive cost | Blocking legitimate content damages creators and trust |
| Multi-modal fusion | Different models per modality; max score or learned fusion |
| Adversarial evasion | Moderation triggers an arms race; build for evolution |

### Common Pitfalls
- Designing one model for all violation types (each has different precision/recall requirements)
- Treating moderation as a static problem
- Ignoring the human review queue as a training data source
- Evaluating only on English content when platform is multilingual
